## Step 1 — Identify the Gold table and saved MLflow model

In [0]:
# inspect your Gold tables
dbutils.widgets.text(
    "catalog",
    "predictivemaintenance_dev",
    "Target Catalog"
)

CATALOG = dbutils.widgets.get("catalog")

display(
    spark.sql(f"SHOW TABLES IN {CATALOG}.gold_data")
)

In [ ]:
# Identify the deployment environment
dbutils.widgets.text("deployment_target", "dev")
DEPLOYMENT_TARGET = dbutils.widgets.get("deployment_target")

# Expected catalog for each environment
EXPECTED_CATALOGS = {
    "dev": "predictivemaintenance_dev",
    "prod": "predictivemaintenancecatalog"
}

# Prevent accidental writes to the wrong catalog
if DEPLOYMENT_TARGET not in EXPECTED_CATALOGS:
    raise ValueError(
        f"Unknown deployment target: {DEPLOYMENT_TARGET}"
    )

if CATALOG != EXPECTED_CATALOGS[DEPLOYMENT_TARGET]:
    raise ValueError(
        f"Catalog {CATALOG} does not match "
        f"deployment target {DEPLOYMENT_TARGET}"
    )

print(f"Deployment target: {DEPLOYMENT_TARGET}")
print(f"Catalog: {CATALOG}")

## Step 2 — Load and test the model

In [0]:
import mlflow

GOLD_TABLE = (
f"{CATALOG}.gold_data.streaming"
)

gold_df = spark.table(GOLD_TABLE)

# Load the selected MLflow model
# MLflow model parameter
dbutils.widgets.text(
    "model_uri",
    "models:/m-2992a2dcaf454e7fb1968eca38e9ed19",
    "MLflow Model URI"
)

MODEL_URI = dbutils.widgets.get("model_uri")

# Load the selected model
model = mlflow.pyfunc.load_model(MODEL_URI)
threshold=float(model.unwrap_python_model().threshold)

# Read the exact feature names from its saved signature
FEATURE_COLS = model.metadata.get_input_schema().input_names()
display(FEATURE_COLS)

missing = [c for c in FEATURE_COLS if c not in gold_df.columns]

assert not missing, f"Missing model features: {missing}"
assert "Machine_failure" not in FEATURE_COLS

print("threshold:", threshold)
print("Gold records:", gold_df.count())
print("Model features:", FEATURE_COLS)
print("Feature validation passed.")

### Test predictions on five observations

In [0]:
sample_pdf = (
    gold_df
    .select(*FEATURE_COLS)
    .limit(5)
    .toPandas()
)

sample_predictions = model.predict(sample_pdf)

display(sample_predictions)

## Step 3 — Stream Gold into the MLflow model

### Create the prediction stream

In [0]:
from pyspark.sql import functions as F



# Turn the saved MLflow model into a Spark UDF
predict_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=MODEL_URI,
    result_type=(
        "struct<failure_probability: double, "
        "predicted_failure: long>"
    )
)

# Read incoming records from Gold
#gold_stream_df = spark.readStream.table(GOLD_TABLE)
gold_stream_df = (
    spark.readStream
    .option("skipChangeCommits", "true")
    .table(GOLD_TABLE)
)


# Keep identifiers for later evaluation
assert "UDI" in gold_stream_df.columns, (
    "Gold must retain UDI to link predictions to test labels."
)

id_cols = [
    c for c in ["UDI", "Product_ID"]
    if c in gold_stream_df.columns
]

# Run model inference
predictions_stream = (
    gold_stream_df
    .withColumn(
        "_model_output",
        predict_udf(
            F.struct(*[F.col(c) for c in FEATURE_COLS])
        )
    )
    .select(
        *id_cols,
        F.col("_model_output.failure_probability")
            .alias("failure_probability"),
        F.col("_model_output.predicted_failure")
            .cast("int")
            .alias("predicted_failure"),
        F.lit(MODEL_URI).alias("model_uri"),
        F.lit(threshold).alias("threshold"),
        F.current_timestamp().alias("prediction_timestamp")
    )
)

### Save predictions into Delta

In [0]:
PREDICTION_TABLE = (
    f"{CATALOG}.gold_data.test_failure_predictions"
)

checkpoint_path = (
    f"/Volumes/{CATALOG}/raw_data/landing/"
    "_checkpoints/33_gold_prediction"
)

query = (
    predictions_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(PREDICTION_TABLE)
)

query.awaitTermination()

print("Prediction stream completed.")

## Validate the results

In [0]:
prediction_df = spark.table(PREDICTION_TABLE)

print("Prediction count:", prediction_df.count())
print("Unique machines:", prediction_df.select("UDI").distinct().count())

display(prediction_df.limit(10))

display(
    prediction_df.groupBy("predicted_failure").count()
)